# 04 — 2026 FIFA World Cup Predictions

Loads trained AutoGluon models, predicts individual matches, and runs Monte Carlo
simulations to estimate group-stage advancement and tournament-winner probabilities.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from autogluon.tabular import TabularPredictor

RAW = Path("../data/raw")
MODELS = Path("../models")

# ── Load models ───────────────────────────────────────────────────────────────
clf = TabularPredictor.load(str(MODELS / "classifier"))
reg = TabularPredictor.load(str(MODELS / "regressor"))

print("Classifier label:", clf.label)
print("Regressor  label:", reg.label)

# ── Load raw data for lookups ─────────────────────────────────────────────────
fifa_rankings = pd.read_csv(RAW / "fifa_rankings.csv")
elo_ratings   = pd.read_csv(RAW / "elo_ratings.csv")
market_values = pd.read_csv(RAW / "market_values.csv")
matches       = pd.read_csv(RAW / "matches.csv", parse_dates=["date"])

# ── Name normalisation (must match 02_feature_engineering.ipynb) ──────────────
NAME_MAP = {
    "Korea Republic":        "South Korea",
    "IR Iran":               "Iran",
    "China PR":              "China",
    "Chinese Taipei":        "Taiwan",
    "Hong Kong, China":      "Hong Kong",
    "Kyrgyz Republic":       "Kyrgyzstan",
    "Macau":                 "Macao",
    "Congo DR":              "DR Congo",
    "Congo, DR":             "DR Congo",
    "Czechia":               "Czech Republic",
    "USA":                   "United States",
    "US Virgin Islands":     "United States Virgin Islands",
    "Cabo Verde":            "Cape Verde",
    "São Tomé and Príncipe": "Sao Tome and Principe",
    "Eswatini":              "Swaziland",
    "Timor-Leste":           "East Timor",
    "Türkiye":               "Turkey",
    "Côte d'Ivoire":         "Ivory Coast",
}

def normalize(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = df[col].replace(NAME_MAP)
    return df

fifa_rankings = normalize(fifa_rankings, ["team"])
elo_ratings   = normalize(elo_ratings,   ["team"])
market_values = normalize(market_values, ["team"])
matches       = normalize(matches,       ["home_team", "away_team"])
matches       = matches.sort_values("date").reset_index(drop=True)

# ── Build lookup dicts ────────────────────────────────────────────────────────
fifa_lookup = {
    row["team"]: {
        "rank":          row["rank"],
        "points":        row["points"],
        "confederation": row["confederation"],
    }
    for _, row in fifa_rankings.iterrows()
}

elo_lookup = {
    row["team"]: row["elo_rating"]
    for _, row in elo_ratings.iterrows()
}

market_lookup = {
    row["team"]: row["squad_total_value_eur"]
    for _, row in market_values.iterrows()
}

print(f"\nfifa_lookup:   {len(fifa_lookup)} teams")
print(f"elo_lookup:    {len(elo_lookup)} teams")
print(f"market_lookup: {len(market_lookup)} teams")

## 2026 FIFA World Cup Groups

Groups are based on the official December 2024 draw.
**Update this cell with any corrections once the final qualified-team lineup is confirmed.**

In [ ]:
# 2026 FIFA World Cup groups — official draw (December 2024).
# Team names must match the normalized names used in the lookup dicts above.
# TODO: update with final draw once all play-off results are confirmed.
GROUPS = {
    "A": ["United States", "Panama", "Morocco", "Portugal"],
    "B": ["Spain",        "New Zealand", "Uzbekistan", "Bahrain"],
    "C": ["Argentina",   "Chile",      "Peru",           "New Caledonia"],
    "D": ["France",      "Saudi Arabia", "Venezuela",   "Ukraine"],
    "E": ["Japan",       "Indonesia",  "Iraq",           "Czech Republic"],
    "F": ["Germany",     "Australia",  "South Korea",    "Cameroon"],
    "G": ["Brazil",      "Dominican Republic", "Egypt", "Serbia"],
    "H": ["England",     "Senegal",    "Slovakia",       "Tunisia"],
    "I": ["Netherlands", "Paraguay",   "Trinidad and Tobago", "Guinea"],
    "J": ["Croatia",     "Bolivia",    "Thailand",       "Cuba"],
    "K": ["Mexico",      "Colombia",   "Ecuador",        "Jamaica"],
    "L": ["Canada",      "Uruguay",    "Algeria",        "Nigeria"],
}

all_wc_teams = [t for teams in GROUPS.values() for t in teams]
print(f"Total teams: {len(all_wc_teams)}")
print(f"Groups: {len(GROUPS)}")

## Feature Generation

Mirrors the logic in `02_feature_engineering.ipynb` but uses current data.
All World Cup matches are `neutral_venue=1`, `match_importance=1.0`.

In [ ]:
DEFAULT_RANK = 100
DEFAULT_PTS  = 1300
DEFAULT_ELO  = 1500
DEFAULT_CONF = "UNK"


def compute_rolling_features(matches_df: pd.DataFrame, team: str, n: int = 10) -> dict:
    """Return form stats for *team* across its last N matches."""
    mask = (matches_df["home_team"] == team) | (matches_df["away_team"] == team)
    team_matches = matches_df[mask].tail(n)

    if team_matches.empty:
        return {"form": 0.5, "goals_scored_avg": 1.0, "goals_conceded_avg": 1.0}

    wins, goals_scored, goals_conceded = 0, 0, 0
    for _, m in team_matches.iterrows():
        if m["home_team"] == team:
            gs, gc = m["home_score"], m["away_score"]
        else:
            gs, gc = m["away_score"], m["home_score"]
        goals_scored   += gs
        goals_conceded += gc
        if gs > gc:
            wins += 1

    n_played = len(team_matches)
    return {
        "form":               wins / n_played,
        "goals_scored_avg":   goals_scored   / n_played,
        "goals_conceded_avg": goals_conceded / n_played,
    }


def compute_h2h(matches_df: pd.DataFrame, home_team: str, away_team: str, n: int = 5) -> float:
    """Return home_team win ratio over the last N head-to-head meetings."""
    mask = (
        ((matches_df["home_team"] == home_team) & (matches_df["away_team"] == away_team)) |
        ((matches_df["home_team"] == away_team) & (matches_df["away_team"] == home_team))
    )
    h2h = matches_df[mask].tail(n)

    if h2h.empty:
        return 0.5

    wins = 0
    for _, m in h2h.iterrows():
        if m["home_team"] == home_team:
            hs, as_ = m["home_score"], m["away_score"]
        else:
            hs, as_ = m["away_score"], m["home_score"]
        if hs > as_:
            wins += 1

    return wins / len(h2h)


def generate_match_features(home_team: str, away_team: str) -> dict:
    """
    Build the feature row for a World Cup match between home_team and away_team.
    Uses current rankings/ELO/market values and rolling form from all historical matches.
    All WC matches are treated as neutral venue with importance=1.0.
    """
    home_form = compute_rolling_features(matches, home_team)
    away_form = compute_rolling_features(matches, away_team)
    h2h_wins  = compute_h2h(matches, home_team, away_team)

    home_fifa = fifa_lookup.get(home_team, {})
    away_fifa = fifa_lookup.get(away_team, {})

    home_rank = home_fifa.get("rank",          DEFAULT_RANK)
    away_rank = away_fifa.get("rank",          DEFAULT_RANK)
    home_pts  = home_fifa.get("points",        DEFAULT_PTS)
    away_pts  = away_fifa.get("points",        DEFAULT_PTS)
    home_conf = home_fifa.get("confederation", DEFAULT_CONF)
    away_conf = away_fifa.get("confederation", DEFAULT_CONF)

    home_elo  = elo_lookup.get(home_team, DEFAULT_ELO)
    away_elo  = elo_lookup.get(away_team, DEFAULT_ELO)

    home_mkt  = market_lookup.get(home_team)
    away_mkt  = market_lookup.get(away_team)
    if home_mkt and away_mkt and away_mkt > 0:
        squad_value_ratio = home_mkt / away_mkt
    else:
        squad_value_ratio = 1.0

    return {
        "tournament":              "FIFA World Cup",
        "elo_diff":                home_elo - away_elo,
        "fifa_rank_diff":          away_rank - home_rank,
        "fifa_points_diff":        home_pts  - away_pts,
        "squad_value_ratio":       squad_value_ratio,
        "form_home":               home_form["form"],
        "form_away":               away_form["form"],
        "goals_scored_avg_home":   home_form["goals_scored_avg"],
        "goals_scored_avg_away":   away_form["goals_scored_avg"],
        "goals_conceded_avg_home": home_form["goals_conceded_avg"],
        "goals_conceded_avg_away": away_form["goals_conceded_avg"],
        "h2h_wins":                h2h_wins,
        "match_importance":        1.0,
        "confederation_home":      home_conf,
        "confederation_away":      away_conf,
        "neutral_venue":           1,
    }


# Smoke-test
sample = generate_match_features("France", "Brazil")
print(pd.Series(sample).to_string())

## Single-Match Predictions

In [ ]:
def predict_match(home_team: str, away_team: str) -> dict:
    """
    Returns:
        p_home_win  : probability home team wins
        p_draw      : probability of a draw
        p_away_win  : probability away team wins
        exp_goal_diff: expected home_score - away_score
    """
    features = generate_match_features(home_team, away_team)
    row = pd.DataFrame([features])

    proba = clf.predict_proba(row)          # DataFrame with columns W, D, L
    exp_gd = float(reg.predict(row).iloc[0])

    return {
        "home_team":      home_team,
        "away_team":      away_team,
        "p_home_win":     float(proba["W"].iloc[0]),
        "p_draw":         float(proba["D"].iloc[0]),
        "p_away_win":     float(proba["L"].iloc[0]),
        "exp_goal_diff":  exp_gd,
    }


showcase_matches = [
    ("France",    "Brazil"),
    ("Argentina", "England"),
    ("Spain",     "Germany"),
    ("United States", "Morocco"),
]

results = [predict_match(h, a) for h, a in showcase_matches]
showcase_df = pd.DataFrame(results)
showcase_df = showcase_df.round(3)
print(showcase_df.to_string(index=False))

## Monte Carlo Group-Stage Simulation

In [ ]:
rng = np.random.default_rng(42)


def _sample_outcome(p_home_win: float, p_draw: float, p_away_win: float) -> str:
    """Sample W / D / L for the home team."""
    r = rng.random()
    if r < p_home_win:
        return "W"
    if r < p_home_win + p_draw:
        return "D"
    return "L"


def _precompute_group_probs(group_teams: list) -> dict:
    """Pre-fetch model probabilities for every pair in the group."""
    probs = {}
    for i, home in enumerate(group_teams):
        for j, away in enumerate(group_teams):
            if i == j:
                continue
            pred = predict_match(home, away)
            probs[(home, away)] = (
                pred["p_home_win"],
                pred["p_draw"],
                pred["p_away_win"],
            )
    return probs


def simulate_group(group_teams: list, n_simulations: int = 10_000) -> dict:
    """
    Run round-robin simulations for a 4-team group.

    Points: win=3, draw=1, loss=0.
    Tiebreaker: goal difference (sampled from regressor expected value + noise).
    Returns {team: probability_of_advancing} for the top-2 teams.
    """
    probs = _precompute_group_probs(group_teams)
    advance_counts = {t: 0 for t in group_teams}

    for _ in range(n_simulations):
        points  = {t: 0   for t in group_teams}
        gd_sim  = {t: 0.0 for t in group_teams}

        for i, home in enumerate(group_teams):
            for j, away in enumerate(group_teams):
                if i >= j:
                    continue

                pw, pd_, pl = probs[(home, away)]
                outcome = _sample_outcome(pw, pd_, pl)

                # Simulate goal difference using expected value + noise
                features = generate_match_features(home, away)
                exp_gd = float(reg.predict(pd.DataFrame([features])).iloc[0])
                noise  = float(rng.normal(0, 1.0))
                sim_gd = exp_gd + noise

                gd_sim[home] += sim_gd
                gd_sim[away] -= sim_gd

                if outcome == "W":
                    points[home] += 3
                elif outcome == "D":
                    points[home] += 1
                    points[away] += 1
                else:
                    points[away] += 3

        # Rank by points then goal difference
        standings = sorted(
            group_teams,
            key=lambda t: (points[t], gd_sim[t]),
            reverse=True,
        )
        # Top 2 advance
        advance_counts[standings[0]] += 1
        advance_counts[standings[1]] += 1

    return {t: advance_counts[t] / n_simulations for t in group_teams}


# Example: Group A
print("Simulating Group A...", flush=True)
group_a_probs = simulate_group(GROUPS["A"], n_simulations=2_000)
print("\nGroup A — P(advance):")
for team, prob in sorted(group_a_probs.items(), key=lambda x: -x[1]):
    print(f"  {team:<25}  {prob:.1%}")

## Full Tournament Simulation

In [ ]:
N_SIM_GROUPS = 5_000   # per group
N_SIM_KO     = 20_000  # for knockout winner estimation


# ── Step 1: simulate every group ─────────────────────────────────────────────
print("Simulating group stage...")
group_advance_probs = {}   # {team: P(advance from group)}

for group_name, teams in GROUPS.items():
    print(f"  Group {group_name}: {teams}")
    group_probs = simulate_group(teams, n_simulations=N_SIM_GROUPS)
    group_advance_probs.update(group_probs)

print("\nGroup stage simulation complete.")


# ── Step 2: estimate tournament-winner probability ────────────────────────────
# Simplified approach: each knockout-round matchup is won with probability
# proportional to the team's ELO rating.  This avoids expanding the full bracket
# (which would require 32-team path enumeration) while still being skill-based.

def elo_win_prob(team_a: str, team_b: str) -> float:
    """P(team_a beats team_b) via ELO difference (no draws in KO rounds)."""
    ea = elo_lookup.get(team_a, DEFAULT_ELO)
    eb = elo_lookup.get(team_b, DEFAULT_ELO)
    return 1 / (1 + 10 ** ((eb - ea) / 400))


def simulate_tournament_winner(
    teams: list,
    advance_probs: dict,
    n_simulations: int = N_SIM_KO,
) -> dict:
    """
    For each simulation:
      1. Sample which teams advance from each group using advance_probs.
      2. Pair advancing teams into a random 48-team knockout bracket
         (simplified: treat all 48 advancers as a pool and run random bracket).
      3. Each match won by ELO-based probability (no draws in KO).
    Returns {team: P(win tournament)}.
    """
    win_counts = {t: 0 for t in teams}

    # Group qualifiers per simulation: 2 teams advance per group
    group_teams_per_group = {
        g: ts for g, ts in GROUPS.items()
    }

    for _ in range(n_simulations):
        # Sample 2 qualifiers from each group
        qualifiers = []
        for group_name, group_ts in group_teams_per_group.items():
            group_advance = [advance_probs.get(t, 0.0) for t in group_ts]
            total = sum(group_advance)
            if total == 0:
                qualified = list(rng.choice(group_ts, size=2, replace=False))
            else:
                weights = [p / total for p in group_advance]
                qualified = list(rng.choice(group_ts, size=2, replace=False, p=weights))
            qualifiers.extend(qualified)

        # Shuffle and run single-elimination bracket
        rng.shuffle(qualifiers)
        bracket = list(qualifiers)

        while len(bracket) > 1:
            next_round = []
            for k in range(0, len(bracket) - 1, 2):
                a, b = bracket[k], bracket[k + 1]
                p_a = elo_win_prob(a, b)
                winner = a if rng.random() < p_a else b
                next_round.append(winner)
            if len(bracket) % 2 == 1:   # bye for odd bracket size
                next_round.append(bracket[-1])
            bracket = next_round

        win_counts[bracket[0]] += 1

    return {t: win_counts[t] / n_simulations for t in teams}


print("\nSimulating knockout stage...")
win_probs = simulate_tournament_winner(
    all_wc_teams, group_advance_probs, n_simulations=N_SIM_KO
)


# ── Results ───────────────────────────────────────────────────────────────────
results_df = pd.DataFrame([
    {
        "team":            t,
        "group":           next(g for g, ts in GROUPS.items() if t in ts),
        "elo":             elo_lookup.get(t, DEFAULT_ELO),
        "p_advance":       group_advance_probs.get(t, 0.0),
        "p_win_wc":        win_probs.get(t, 0.0),
    }
    for t in all_wc_teams
]).sort_values("p_win_wc", ascending=False).reset_index(drop=True)

results_df["p_advance"] = results_df["p_advance"].map("{:.1%}".format)
results_df["p_win_wc"]  = results_df["p_win_wc"].map("{:.2%}".format)

print("\n=== Top 15 Teams by Tournament Win Probability ===")
print(results_df[["team", "group", "elo", "p_advance", "p_win_wc"]].head(15).to_string(index=False))